# 🚌 대중교통·날씨 맞춤형 AI 추론 모델 — Colab 학습 파이프라인 (v2.1)

> **목적**: `docs/AI_MODEL_SPEC.md` §3.2의 **서빙 입력 계약 32차원**과 `main.py::build_feature_vector`의 **피처 순서(index 0~31)를 문자 단위로 일치**시키는 실서빙용 모델을 학습합니다.

| 항목 | 내용 |
| :--- | :--- |
| **원천 데이터** | 2023~2025 3개년 통합데이터 (`data/통합데이터/` 63만 건 × 28열) 또는 2025 전처리_2 CSV |
| **데이터 분할** | 3개년 분할 (`2023~2024` Train/Val 42.0만 건 / `2025` 1년 전체 Test 20.9만 건) |
| **공휴일 캘린더** | 2023~2025 대한민국 공휴일 (`main.py::KR_HOLIDAYS` 와 완전 동일 집합) |
| **입력 차원** | **32차원 (실제 피처 32개)** — 28열 신규 포맷(지하철_하차/풍향 등 5열 삭제) 호환 완비 |
| **산출물** | `lgb_{bike,bus,subway}_demand.onnx` (32차원) + `feature_schema.json` |

### ✅ 핵심 사양 정합 (2026-10-29 최신화)
1. **28열 신규 포맷 호환**: `지하철_하차`, `풍향` 등 미사용 5개 열 삭제를 완벽히 지원.
2. **2023~2025 3개년 다년도 학습**: 42만 행 대규모 학습 및 2025년 전체 1년 일반화 검증.
3. **공간 인프라 4 + `same_day_last_week` 3 = 실제 피처로 채움**
4. **`is_holiday` = 대한민국 공휴일 캘린더**: 주말 지표와 분리하여 실측 반영.
5. **`wind_speed` & `rain_intensity` 경계 완벽 정합**


In [1]:
# [Cell 1] 구글 드라이브 마운트 + 학습 의존성
# Colab 기본 제공이 아닌 패키지는 각 셀에서 자동 설치됩니다 (optuna → Cell 6-1, onnxmltools/skl2onnx → Cell 7).
# 네트워크가 막힌 환경에서는 아래 주석을 해제해 1회 수동 설치하세요.
# !pip install -q lightgbm optuna onnxmltools skl2onnx onnxruntime

from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


---
### [Step 2] 원천 데이터 로드 및 결측치 도메인 정제
`2025_서울25개구_교통날씨_전처리_2.csv` (209,875행 × 33열)를 로드합니다.
이 파일은 캘린더 파생(`월`/`요일`/`주말여부`/`연중일`), 결측 플래그(`*_결측`), 공식 분할(`분할`) 컬럼을 포함하지만
**모델 입력 피처(32차원)는 포함하지 않으므로** 이후 단계에서 직접 엔지니어링합니다.


In [3]:
# [Cell 3] 데이터 로드 및 결측치 정제 (3개년 28열 통합데이터 또는 2025 단일 CSV 자동 감지)
import os
from pathlib import Path
import numpy as np
import pandas as pd

# ── 1. 구글 드라이브 경로 탐색 (신규 다년도 vs 레거시 자동 감지) ──────────────────────
MULTI_YEAR_DIR = Path('/content/drive/MyDrive/weather-transit-web/data/통합데이터')
LEGACY_CSV_PATH = Path('/content/drive/MyDrive/weather-transit-web/data/2025_서울25개구_교통날씨_전처리_2.csv')

if MULTI_YEAR_DIR.exists() and (MULTI_YEAR_DIR / '25_통합데이터/2025_통합데이터_test.csv').exists():
    print("🚀 [모드] 구글 드라이브 2023~2025 3개년 통합데이터(28열) 로드 중...")
    df23 = pd.read_csv(MULTI_YEAR_DIR / '23_통합데이터/2023_통합데이터_train.csv')
    df24 = pd.read_csv(MULTI_YEAR_DIR / '24_통합데이터/2024_통합데이터_train.csv')
    df25 = pd.read_csv(MULTI_YEAR_DIR / '25_통합데이터/2025_통합데이터_test.csv')
    df = pd.concat([df23, df24, df25], ignore_index=True)
    DATASET_MODE = 'multiyear_28col'
elif LEGACY_CSV_PATH.exists():
    print("⚠️ [모드] 구글 드라이브 2025 전처리_2 단일 파일(33열) 로드 중...")
    df = pd.read_csv(LEGACY_CSV_PATH)
    DATASET_MODE = 'legacy_2025'
else:
    # 로컬 경로 또는 Colab 현재 세션 업로드 경로 탐색
    found = list(Path('.').glob('**/*2025*test.csv')) or list(Path('.').glob('**/*전처리_2.csv'))
    if found:
        print(f"📂 로컬 탐색 경로 로드: {found[0]}")
        df = pd.read_csv(found[0])
        DATASET_MODE = 'local_fallback'
    else:
        raise FileNotFoundError(
            "구글 드라이브에 데이터가 없습니다! "
            "MyDrive/weather-transit-web/data/통합데이터/ 폴더 또는 2025 전처리 CSV를 확인하세요."
        )

print(f"원천 데이터 로드 완료: {df.shape} (모드: {DATASET_MODE})")

# ── 2. 결측치 및 도메인 정제 ────────────────────────────────────────────────────────
# 지하철 심야 운휴(01~05시) 명시적 0 처리 (지하철_하차 열 삭제 대응 방어 로직)
subway_cols = [c for c in ['지하철_승차', '지하철_하차'] if c in df.columns]
if subway_cols:
    df.loc[df['시간'].isin([1, 2, 3, 4, 5]), subway_cols] = 0

# 기상 결측 보간 (강수는 0 대체, 기온/풍속/습도는 선형 보간)
df['강수량(mm)'] = df['강수량(mm)'].fillna(0.0)
for c in ['기온(°C)', '풍속(m/s)', '습도(%)']:
    if c in df.columns:
        df[c] = df[c].interpolate(method='linear')

# 타깃 결측 보정
df['버스_승차'] = df['버스_승차'].interpolate(method='linear').bfill().ffill()
df['따릉이_대여'] = df['따릉이_대여'].fillna(0)

print("타깃 잔여 결측:", int(df[['따릉이_대여', '버스_승차', '지하철_승차']].isna().sum().sum()))
if '분할' in df.columns:
    print("데이터셋 분할 분포:\n", df['분할'].value_counts().to_string())

🚀 [모드] 구글 드라이브 2023~2025 3개년 통합데이터(28열) 로드 중...
원천 데이터 로드 완료: (630200, 28) (모드: multiyear_28col)
타깃 잔여 결측: 0
데이터셋 분할 분포:
 분할
train    420325
test     209875


---
### [Step 3] 특성 공학 — **실제 32차원** (사양 §3.2 순서)
`main.py::build_feature_vector`가 만드는 벡터와 **동일한 순서**로 32개 피처를 생성합니다.

| 블록 | 인덱스 | 차원 | 내용 |
| :--- | :--- | :---: | :--- |
| 1. 시계열·캘린더 | `0..7` | 8 | month/hour sin·cos, dayofweek, is_weekend, **is_holiday**, is_rush_hour |
| 2. 기상 관측·파생 | `8..15` | 8 | temperature, wind_speed, humidity, precipitation, is_raining, rain_intensity, sensible_temp, discomfort_index |
| 3. 공간(위치) 인코딩 | `16..20` | 5 | district_code, **district_subway_stations**, **district_bus_stops**, **district_bike_racks**, **commercial_density** |
| 4. 시계열 래그·이동통계 | `21..31` | 11 | lag_1h ×3, roll_3h ×3, **same_day_last_week ×3**, temp_diff_1h, rain_diff_1h |


In [4]:
# [Cell 4-A] 블록 1~3 생성 (캘린더 8 + 기상 8 + 공간 5 = 21차원)
df['기준_날짜'] = pd.to_datetime(df['기준_날짜'])
df['기준_일시'] = pd.to_datetime(df['기준_일시'])

# 구/시간 오름차순 정렬 (래그·이동통계 산출의 전제)
df = df.sort_values(['구', '기준_일시']).reset_index(drop=True)

# ── 블록 1. 시계열 및 캘린더 피처 (8) ────────────────────────────────
df['month'] = df['기준_날짜'].dt.month
df['dayofweek'] = df['기준_날짜'].dt.dayofweek
df['is_weekend'] = (df['dayofweek'] >= 5).astype(int)
# ⚠️ 정합 필수: is_holiday 는 main.py::KR_HOLIDAYS 와 완전히 동일한 공휴일 집합을 사용합니다.
#    2025 대한민국 공휴일 19일 — 신정 / 설날 연휴(임시공휴일 01-27 포함) / 삼일절+대체 /
#    어린이날·부처님오신날+대체 / 제21대 대선 / 현충일 / 광복절 / 개천절 /
#    추석 연휴+대체 / 한글날 / 성탄절. (노동절·제헌절은 2025년 공휴일 아님)
KR_HOLIDAYS_2025 = {
    '2025-01-01',                                            # 신정
    '2025-01-27',                                            # 임시공휴일(설 연휴)
    '2025-01-28', '2025-01-29', '2025-01-30',                # 설날 연휴
    '2025-03-01', '2025-03-03',                              # 삼일절 + 대체공휴일
    '2025-05-05', '2025-05-06',                              # 어린이날·부처님오신날 + 대체공휴일
    '2025-06-03',                                            # 제21대 대통령선거일
    '2025-06-06',                                            # 현충일
    '2025-08-15',                                            # 광복절
    '2025-10-03',                                            # 개천절
    '2025-10-05', '2025-10-06', '2025-10-07', '2025-10-08',   # 추석 연휴 + 대체공휴일
    '2025-10-09',                                            # 한글날
    '2025-12-25',                                            # 성탄절
}
df['is_holiday'] = df['기준_날짜'].dt.strftime('%Y-%m-%d').isin(KR_HOLIDAYS_2025).astype(int)
# is_weekend(토·일)와는 별개 지표입니다(주말이면서 공휴일인 날은 둘 다 1).
assert 0.01 < df['is_holiday'].mean() < 0.20, f"공휴일 비율 비정상: {df['is_holiday'].mean():.4f}"
df['is_rush_hour'] = df['시간'].isin([8, 9, 18, 19]).astype(int)

df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12.0)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12.0)
df['hour_sin'] = np.sin(2 * np.pi * df['시간'] / 24.0)
df['hour_cos'] = np.cos(2 * np.pi * df['시간'] / 24.0)

# ── 블록 2. 기상 관측 및 파생 피처 (8) ───────────────────────────────
df['temperature'] = df['기온(°C)']
df['wind_speed'] = df['풍속(m/s)'].clip(lower=0.1)   # 서빙: max(wind, 0.1)
df['humidity'] = df['습도(%)']
df['precipitation'] = df['강수량(mm)']
df['is_raining'] = (df['precipitation'] > 0.0).astype(int)

# 서빙과 동일한 경계: <=0 → 0, <3 → 1, <=10 → 2, 그 외 → 3
df['rain_intensity'] = np.select(
    [df['precipitation'] <= 0.0, df['precipitation'] < 3.0, df['precipitation'] <= 10.0],
    [0, 1, 2], default=3,
).astype(int)

_v = df['wind_speed'] ** 0.16
df['sensible_temp'] = 13.12 + 0.6215 * df['temperature'] - 11.37 * _v + 0.3965 * df['temperature'] * _v
df['discomfort_index'] = (
    1.8 * df['temperature'] - 0.55 * (1 - df['humidity'] / 100.0) * (1.8 * df['temperature'] - 26.0) + 32.0
)

# ── 블록 3. 공간(위치) 인코딩 피처 (5) ───────────────────────────────
# district_code 는 main.py::DISTRICT_CODE_MAP 과 순서를 반드시 동일하게 고정합니다.
DISTRICT_CODE_MAP = {
    '강남구': 0, '강동구': 1, '강북구': 2, '강서구': 3, '관악구': 4,
    '광진구': 5, '구로구': 6, '금천구': 7, '노원구': 8, '도봉구': 9,
    '동대문구': 10, '동작구': 11, '마포구': 12, '서대문구': 13, '서초구': 14,
    '성동구': 15, '성북구': 16, '송파구': 17, '양천구': 18, '영등포구': 19,
    '용산구': 20, '은평구': 21, '종로구': 22, '중구': 23, '중랑구': 24,
}
df['district_code'] = df['구'].map(DISTRICT_CODE_MAP).astype(int)

# 25개 자치구 실측 인프라 (main.py::DISTRICT_INFRA 전수 동기화)
DISTRICT_INFRA = {
    '강남구': (34.0, 552.0, 172.0, 1.48), '강동구': (16.0, 398.0, 116.0, 0.86),
    '강북구': (13.0, 428.0, 90.0, 0.72),  '강서구': (23.0, 628.0, 212.0, 0.92),
    '관악구': (9.0, 492.0, 132.0, 0.88),  '광진구': (11.0, 298.0, 120.0, 0.96),
    '구로구': (15.0, 524.0, 138.0, 1.08), '금천구': (5.0, 368.0, 96.0, 1.02),
    '노원구': (18.0, 558.0, 158.0, 0.82), '도봉구': (8.0, 402.0, 102.0, 0.70),
    '동대문구': (16.0, 498.0, 130.0, 1.05), '동작구': (17.0, 448.0, 118.0, 0.92),
    '마포구': (18.0, 574.0, 168.0, 1.28), '서대문구': (10.0, 462.0, 110.0, 0.95),
    '서초구': (27.0, 615.0, 156.0, 1.36), '성동구': (17.0, 442.0, 128.0, 1.12),
    '성북구': (14.0, 602.0, 124.0, 0.85), '송파구': (30.0, 645.0, 228.0, 1.22),
    '양천구': (10.0, 388.0, 136.0, 0.86), '영등포구': (21.0, 516.0, 184.0, 1.42),
    '용산구': (15.0, 356.0, 106.0, 1.18), '은평구': (16.0, 518.0, 142.0, 0.76),
    '종로구': (19.0, 386.0, 126.0, 1.35), '중구': (26.0, 248.0, 114.0, 1.46),
    '중랑구': (12.0, 378.0, 122.0, 0.76),
}
infra_df = pd.DataFrame(
    DISTRICT_INFRA,
    index=['district_subway_stations', 'district_bus_stops', 'district_bike_racks', 'commercial_density'],
).T

assert len(DISTRICT_INFRA) == 25, "인프라 테이블은 25개 자치구 전수여야 합니다."
assert set(df['구'].unique()) == set(DISTRICT_INFRA.keys()), "CSV 자치구명과 인프라 키가 불일치합니다."

df = df.merge(infra_df, left_on='구', right_index=True, how='left')
assert df['district_subway_stations'].notna().all(), "인프라 조인 누락"
print(f"블록 1~3 생성 완료: {df.shape} | 인프라 4종 조인 정상(누락 0)")


블록 1~3 생성 완료: (630200, 50) | 인프라 4종 조인 정상(누락 0)


In [5]:
# [Cell 4-B] 블록 4 생성 (시계열 래그·이동통계 11) + 최종 32차원 확정
LAG_TARGETS = {'bike': '따릉이_대여', 'bus': '버스_승차', 'sub': '지하철_승차'}
N_ROWS_BEFORE = len(df)

# ── 1시간 래그 및 3시간 이동평균 ─────────────────────────────────────
for mode, col in LAG_TARGETS.items():
    df[f'{col}_lag_1h'] = df.groupby('구')[col].shift(1)
    df[f'{col}_roll_3h'] = df.groupby('구')[col].transform(lambda x: x.rolling(3, min_periods=1).mean())

# 구별 첫 관측치(01-01 01시)의 lag 결측 → 3시간 이동평균으로 보정
for col in LAG_TARGETS.values():
    df[f'{col}_lag_1h'] = df[f'{col}_lag_1h'].fillna(df[f'{col}_roll_3h'])

# ── 기상 변화량 ──────────────────────────────────────────────────────
df['temp_diff_1h'] = df.groupby('구')['temperature'].diff().fillna(0.0)
df['rain_diff_1h'] = df.groupby('구')['precipitation'].diff().fillna(0.0)

# ── same_day_last_week (지난주 동요일·동시간) ────────────────────────
# 주의: 하루 23개 시간대(01~23시)이므로 "7일 전 = 168행"이 아니라 161행입니다.
#       따라서 행 위치 shift 가 아닌, 7일 전 타임스탬프 기준 self-merge 로 계산합니다.
prev = df[['구', '기준_일시'] + list(LAG_TARGETS.values())].copy()
prev['기준_일시'] = prev['기준_일시'] + pd.Timedelta(days=7)
prev = prev.rename(columns={
    '따릉이_대여': 'same_day_last_week_bike',
    '버스_승차': 'same_day_last_week_bus',
    '지하철_승차': 'same_day_last_week_sub',
})
df = df.merge(prev, on=['구', '기준_일시'], how='left')
assert len(df) == N_ROWS_BEFORE, "same_day_last_week 조인으로 행 수가 변했습니다(키 중복 확인 필요)."

n_missing = int(df['same_day_last_week_bike'].isna().sum())
for mode, col in LAG_TARGETS.items():
    dst = f'same_day_last_week_{mode}'
    df[dst] = df[dst].fillna(df[f'{col}_lag_1h']).fillna(0.0)
print(f"[same_day_last_week] 7일 이전 데이터 없음 → 1시간 전 값으로 프록시 대체: {n_missing:,}건 "
      f"({n_missing / len(df) * 100:.2f}%, 2025-01-01~01-07 구간)")

# ── 최종 32차원 피처 순서 (main.py::build_feature_vector 인덱스 0~31 과 완전 일치) ──
FINAL_FEATURE_COLS = [
    # [0..7] 시계열 및 캘린더 피처
    'month_sin', 'month_cos', 'hour_sin', 'hour_cos', 'dayofweek', 'is_weekend', 'is_holiday', 'is_rush_hour',
    # [8..15] 기상 관측 및 파생 피처
    'temperature', 'wind_speed', 'humidity', 'precipitation',
    'is_raining', 'rain_intensity', 'sensible_temp', 'discomfort_index',
    # [16..20] 공간(위치) 인코딩 피처
    'district_code', 'district_subway_stations', 'district_bus_stops',
    'district_bike_racks', 'commercial_density',
    # [21..31] 시계열 래그 및 이동통계 피처
    '따릉이_대여_lag_1h', '버스_승차_lag_1h', '지하철_승차_lag_1h',
    '따릉이_대여_roll_3h', '버스_승차_roll_3h', '지하철_승차_roll_3h',
    'same_day_last_week_bike', 'same_day_last_week_bus', 'same_day_last_week_sub',
    'temp_diff_1h', 'rain_diff_1h',
]

assert len(FINAL_FEATURE_COLS) == 32, f"32차원이 아닙니다: {len(FINAL_FEATURE_COLS)}"
assert len(set(FINAL_FEATURE_COLS)) == 32, "중복 피처명이 있습니다."
assert int(df[FINAL_FEATURE_COLS].isna().sum().sum()) == 0, "피처에 결측치가 남아 있습니다."
print(f"최종 피처 확정: {len(FINAL_FEATURE_COLS)}차원 | 결측 0 | 순서 = 사양 §3.2 (8/8/5/11)")


[same_day_last_week] 7일 이전 데이터 없음 → 1시간 전 값으로 프록시 대체: 4,025건 (0.64%, 2025-01-01~01-07 구간)
최종 피처 확정: 32차원 | 결측 0 | 순서 = 사양 §3.2 (8/8/5/11)


---
### [Step 4] 시간 순서 기반 데이터셋 분할 (Temporal Split)
**사양서(`docs/AI_MODEL_SPEC.md` §6.1)의 경계를 그대로 사용합니다** (Data Leakage 차단).

| 분할 | 기간 | 행 수 |
| :--- | :--- | ---: |
| `train` | 2025-01-01 ~ 2025-09-30 | 156,975 |
| `valid` | 2025-10-01 ~ 2025-11-15 | 26,450 |
| `test` | 2025-11-16 ~ 2025-12-31 | 26,450 |

> ⚠️ 전처리 CSV의 `분할` 컬럼은 `valid 10-01~10-31` / `test 11-01~12-31` 로 **사양서와 경계가 다릅니다.**
> 데이터팀 협의 결과 **사양서 §6.1 경계를 공식 기준으로 확정**했으므로 `SPLIT_SOURCE = 'spec_dates'` 를 기본값으로 사용합니다.
> (`'csv_column'` 은 과거 결과 재현용으로만 남겨둡니다.)


In [6]:
# [Cell 5] 데이터셋 분할 (다년도 데이터 vs 단년도 사양서 §6.1 자동 분기)
is_multiyear = ('분할' in df.columns and set(df['분할'].dropna().unique()) >= {'train', 'test'})

if is_multiyear:
    print("📊 [분할 모드] 2023~2025 3개년 분할 (2023~2024 Train/Val -> 2025 1년 전체 Test)")
    # Train: 2023 전체 + 2024 1~9월 (약 35.6만 건)
    train_mask = (df['분할'] == 'train') & (df['기준_날짜'] < '2024-10-01')
    # Validation: 2024 10~12월 (약 6.4만 건)
    val_mask = (df['분할'] == 'train') & (df['기준_날짜'] >= '2024-10-01')
    # Test: 2025 1년 전체 (20.9만 건)
    test_mask = (df['분할'] == 'test')
    SPLIT_SOURCE = 'multiyear_2023_2025'
else:
    print("📊 [분할 모드] 단년도 2025 Temporal Split (사양서 §6.1 공식 기준)")
    train_mask = df['기준_날짜'] <= '2025-09-30'
    val_mask = (df['기준_날짜'] > '2025-09-30') & (df['기준_날짜'] <= '2025-11-15')
    test_mask = df['기준_날짜'] > '2025-11-15'
    SPLIT_SOURCE = 'spec_dates'

X_train = df.loc[train_mask, FINAL_FEATURE_COLS].astype(np.float32).reset_index(drop=True)
X_val = df.loc[val_mask, FINAL_FEATURE_COLS].astype(np.float32).reset_index(drop=True)
X_test = df.loc[test_mask, FINAL_FEATURE_COLS].astype(np.float32).reset_index(drop=True)

for name, mask, X in [('train', train_mask, X_train), ('valid', val_mask, X_val), ('test', test_mask, X_test)]:
    d = df.loc[mask, '기준_날짜']
    print(f"{name:5s} | n={len(X):>7,} | {X.shape[1]}차원 | {d.min().date()} ~ {d.max().date()}")


📊 [분할 모드] 2023~2025 3개년 분할 (2023~2024 Train/Val -> 2025 1년 전체 Test)
train | n=367,425 | 32차원 | 2023-01-01 ~ 2024-09-30
valid | n= 52,900 | 32차원 | 2024-10-01 ~ 2024-12-31
test  | n=209,875 | 32차원 | 2025-01-01 ~ 2025-12-31


---
### [Step 5] 베이스라인 LightGBM 학습 및 평가
타깃 3개(`따릉이_대여`, `버스_승차`, `지하철_승차`)에 대해 **32차원 입력**으로 독립 회귀 모델을 학습합니다.
Optuna 튜닝(Step 5-1) 결과와 비교하기 위한 기준선입니다.


In [7]:
# [Cell 6] 베이스라인 학습 및 성능 검증 (32차원)
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

TARGETS = {'bike': '따릉이_대여', 'bus': '버스_승차', 'subway': '지하철_승차'}
trained_models = {}
baseline_metrics = {}

for mode, target_col in TARGETS.items():
    y_tr = df.loc[train_mask, target_col].astype(np.float64).to_numpy()
    y_va = df.loc[val_mask, target_col].astype(np.float64).to_numpy()
    y_te = df.loc[test_mask, target_col].astype(np.float64).to_numpy()

    model = lgb.LGBMRegressor(
        objective='regression', metric='rmse', boosting_type='gbdt',
        n_estimators=1000, learning_rate=0.03, num_leaves=63, max_depth=8,
        subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1, verbose=-1,
    )
    model.fit(
        X_train, y_tr,
        eval_set=[(X_val, y_va)],
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)],
    )

    preds = np.clip(model.predict(X_test), 0, None)
    baseline_metrics[mode] = {
        'r2': r2_score(y_te, preds),
        'rmse': mean_squared_error(y_te, preds) ** 0.5,
        'mae': mean_absolute_error(y_te, preds),
        'best_iteration': int(getattr(model, 'best_iteration_', 0) or model.n_estimators),
    }
    m = baseline_metrics[mode]
    print(f"[{mode.upper()}] R²={m['r2']:.4f} | RMSE={m['rmse']:,.2f} | MAE={m['mae']:,.2f} "
          f"| best_iteration={m['best_iteration']}")
    trained_models[mode] = model


[BIKE] R²=0.9889 | RMSE=22.97 | MAE=12.59 | best_iteration=1000
[BUS] R²=0.9958 | RMSE=366.35 | MAE=230.91 | best_iteration=1000
[SUBWAY] R²=0.9959 | RMSE=592.77 | MAE=257.48 | best_iteration=1000


---
### [Step 5-1] Optuna 하이퍼파라미터 튜닝 및 최종 학습
타깃별로 베이지안 최적화(Trial 30회, valid RMSE 최소화) 후, 최적 파라미터로 `n_estimators=1500` + early stopping 재학습하여 **실서빙용 모델**을 확정합니다.


In [ ]:
# [Cell 6-1] Optuna 튜닝 → 최종 모델 학습 (32차원)
# Colab 기본 제공이 아닌 패키지는 자동 설치 (optuna 미설치 환경 대응)
try:
    import optuna
except ModuleNotFoundError:
    !pip install -q optuna
    import importlib
    importlib.invalidate_caches()  # 같은 커널에서 방금 설치한 패키지 즉시 인식
    import optuna
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

N_TRIALS = 30
FINAL_N_ESTIMATORS = 1500

tuned_models = {}
tuned_metrics = {}

for mode, target_col in TARGETS.items():
    print(f"\n================ [{mode.upper()}] Optuna Tuning Start ================")
    y_tr = df.loc[train_mask, target_col].astype(np.float64).to_numpy()
    y_va = df.loc[val_mask, target_col].astype(np.float64).to_numpy()
    y_te = df.loc[test_mask, target_col].astype(np.float64).to_numpy()

    def objective(trial, y_tr=y_tr, y_va=y_va):
        params = {
            'objective': 'regression', 'metric': 'rmse', 'boosting_type': 'gbdt',
            'n_estimators': 500,  # 빠른 탐색
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
            'num_leaves': trial.suggest_int('num_leaves', 31, 127),
            'max_depth': trial.suggest_int('max_depth', 6, 10),
            'subsample': trial.suggest_float('subsample', 0.6, 0.9),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 0.9),
            'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
            'random_state': 42, 'n_jobs': -1, 'verbose': -1,
        }
        model = lgb.LGBMRegressor(**params)
        model.fit(X_train, y_tr, eval_set=[(X_val, y_va)],
                  callbacks=[lgb.early_stopping(stopping_rounds=30, verbose=False)])
        preds = model.predict(X_val)
        return float(np.sqrt(np.mean((y_va - preds) ** 2)))

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=N_TRIALS, n_jobs=1)
    print(f"[{mode.upper()} Best Trial] valid RMSE={study.best_value:,.4f}")
    print(f"Best Params: {study.best_params}")

    best_params = dict(study.best_params)
    best_params.update({
        'objective': 'regression', 'metric': 'rmse', 'boosting_type': 'gbdt',
        'n_estimators': FINAL_N_ESTIMATORS, 'random_state': 42, 'n_jobs': -1, 'verbose': -1,
    })
    final_model = lgb.LGBMRegressor(**best_params)
    final_model.fit(X_train, y_tr, eval_set=[(X_val, y_va)],
                    callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)])

    preds = np.clip(final_model.predict(X_test), 0, None)
    tuned_metrics[mode] = {
        'r2': r2_score(y_te, preds),
        'rmse': mean_squared_error(y_te, preds) ** 0.5,
        'mae': mean_absolute_error(y_te, preds),
        'best_params': study.best_params,
        'best_iteration': int(getattr(final_model, 'best_iteration_', 0) or FINAL_N_ESTIMATORS),
    }
    t, b = tuned_metrics[mode], baseline_metrics[mode]
    print(f"[{mode.upper()} 최종] R²={t['r2']:.4f} | RMSE={t['rmse']:,.2f} | MAE={t['mae']:,.2f} "
          f"| best_iteration={t['best_iteration']}")
    print(f"[{mode.upper()} 개선] ΔR²={t['r2'] - b['r2']:+.4f} | RMSE {(t['rmse'] - b['rmse']) / b['rmse'] * 100:+.2f}%")
    tuned_models[mode] = final_model



================ [BIKE] Optuna Tuning Start ================
[BIKE Best Trial] valid RMSE=23.4360
Best Params: {'learning_rate': 0.09737213718467148, 'num_leaves': 115, 'max_depth': 10, 'subsample': 0.6042083739397852, 'colsample_bytree': 0.8354334394368443, 'reg_alpha': 4.417464355796284, 'reg_lambda': 0.38728221291236736}


---
### [Step 6] ONNX 포팅 및 로컬 다운로드
Optuna 최종 모델을 **32차원 고정 입력 스펙**(`FloatTensorType([None, 32])`)으로 ONNX(Opset 14) 변환합니다.
피처 순서를 서버 코드와 맞추기 위해 `feature_schema.json`도 함께 저장합니다.


In [ ]:
# [Cell 7] ONNX 변환 (32차원 고정) + 피처 스키마 저장 + 자동 다운로드
import json
import os

# Colab 기본 제공이 아닌 패키지는 자동 설치 (onnxmltools/skl2onnx/onnxruntime 미설치 환경 대응)
try:
    import onnxmltools
    import onnxruntime as ort
except ModuleNotFoundError:
    !pip install -q onnxmltools skl2onnx onnxruntime
    import importlib
    importlib.invalidate_caches()  # 같은 커널에서 방금 설치한 패키지 즉시 인식
    import onnxmltools
    import onnxruntime as ort
from onnxmltools.convert.common.data_types import FloatTensorType
from google.colab import files

OUT_DIR = '/content/models'
os.makedirs(OUT_DIR, exist_ok=True)

FEATURE_DIM = len(FINAL_FEATURE_COLS)
assert FEATURE_DIM == 32, f"ONNX 입력 차원은 32 여야 합니다: {FEATURE_DIM}"
initial_type = [('float_input', FloatTensorType([None, FEATURE_DIM]))]

# 피처 스키마 (서빙 코드 main.py::build_feature_vector 와 순서 일치 검증용)
schema = {
    'model_version': 'v2.1-multiyear-32d' if is_multiyear else 'v2-전처리_2-32d',
    'feature_dim': FEATURE_DIM,
    'feature_cols': FINAL_FEATURE_COLS,
    'order': ['calendar(8)', 'weather(8)', 'spatial(5)', 'timeseries(11)'],
    'targets': TARGETS,
    'split': {'train': '2023-01-01~2024-09-30', 'valid': '2024-10-01~2024-12-31', 'test': '2025-01-01~2025-12-31'} if is_multiyear else {'train': '2025-01-01~2025-09-30', 'valid': '2025-10-01~2025-11-15', 'test': '2025-11-16~2025-12-31'},
    'split_source': SPLIT_SOURCE,
    'holiday_calendar': 'KR_HOLIDAYS_2025 (19일, main.py::KR_HOLIDAYS 와 동일)',
    'metrics': {m: {'baseline': baseline_metrics[m], 'tuned': tuned_metrics[m]} for m in TARGETS},
}
with open(f'{OUT_DIR}/feature_schema.json', 'w', encoding='utf-8') as f:
    json.dump(schema, f, ensure_ascii=False, indent=2)

onnx_paths = []
for mode, model in tuned_models.items():
    save_path = f'{OUT_DIR}/lgb_{mode}_demand.onnx'
    onnx_model = onnxmltools.convert_lightgbm(model, initial_types=initial_type, target_opset=14)
    with open(save_path, 'wb') as f:
        f.write(onnx_model.SerializeToString())
    print(f"[ONNX 저장 완료] {save_path} ({os.path.getsize(save_path) / 1024:.1f} KB)")
    onnx_paths.append(save_path)

# 무결성 검증: 입력 차원이 32로 고정되었는지 확인
for p in onnx_paths:
    sess = ort.InferenceSession(p, providers=['CPUExecutionProvider'])
    print(f"  - {os.path.basename(p)}: input_dim={sess.get_inputs()[0].shape[1]}")

for path in onnx_paths + [f'{OUT_DIR}/feature_schema.json']:
    files.download(path)

---
### [Step 7] 로컬 데스크탑 프로젝트 반영
1. 다운로드된 3개 파일을 프로젝트 `models/` 폴더에 넣습니다.
   - `lgb_bike_demand.onnx`, `lgb_bus_demand.onnx`, `lgb_subway_demand.onnx`
2. `feature_schema.json`은 참고용으로 `benchmarks/` 하위에 보관합니다.
3. 서버는 `main.py::prepare_session_input`이 **입력 차원 32를 자동 감지**하여 전체 32차원 벡터를 그대로 전달합니다(더미 패딩 불필요).
4. 스모크 테스트:
   ```bash
   python -c "import onnxruntime as ort; [print(p, ort.InferenceSession(f'models/{p}').get_inputs()[0].shape) for p in ['lgb_bike_demand.onnx','lgb_bus_demand.onnx','lgb_subway_demand.onnx']]"
   uvicorn main:app --reload
   ```

### ✅ 서빙 정합 체크리스트 (`main.py::build_feature_vector` 대조)
| # | 항목 | 학습(본 노트북) | 서빙(main.py) | 일치 |
| :--: | :--- | :--- | :--- | :--: |
| 1 | 피처 개수/순서 | 32차원 8/8/5/11 | `vector_32` 인덱스 0~31 | ✅ |
| 2 | `is_holiday` | `KR_HOLIDAYS_2025` (19일 캘린더) | `is_kr_public_holiday(dt)` | ✅ |
| 3 | `wind_speed` | `clip(lower=0.1)` | `max(wind, 0.1)` | ✅ |
| 4 | `rain_intensity` | `<=0→0, <3→1, <=10→2, else→3` | 동일 | ✅ |
| 5 | 인프라 4종 | `DISTRICT_INFRA` 25개구 | 동일 테이블 | ✅ |
| 6 | `same_day_last_week` | 7일 전 동시간 self-merge | 시계열 캐시 동일 정의 | ✅ |
| 7 | `district_code` | `DISTRICT_CODE_MAP` 고정 순서 | 동일 맵 | ✅ |
